In [1]:
# Parameters
DB_PATH                   = "../../../DB/oedb_refiner_1st.db"
BENCHMARK_PATH            = "../../../data/input_data/benchmark_trainingset.xlsx"
BENCHMARK_SHEET           = "merged_answers"
MATCHED_ANSWERS_CSV       = "matched_answers.csv"
MATCHED_PARTICIPANTS_CSV  = "matched_participants.csv"
MATCHED_QUESTIONS_CSV     = "matched_questions.csv"
NOTEGROUP_ID_MIN          = 1
NOTEGROUP_ID_MAX          = 23

EVAL_FIELDS = ["participantID", "questionID", "answer_content_oriLAN", "answer_content_EN"]
CONTENT_SIM_THRESHOLD = 0.95   # ratio scale 0-1, fuzz.ratio returns 0-100

In [2]:
import sqlite3
import re
import pandas as pd
from rapidfuzz import fuzz

def load_etl(db_path, id_min, id_max):
    con = sqlite3.connect(db_path)
    df = pd.read_sql_query(
        """SELECT answerID, notegroupID, questionID, participantID,
                  answer_content_oriLAN, answer_content_EN
           FROM answers
           WHERE notegroupID BETWEEN ? AND ?""",
        con, params=(id_min, id_max)
    )
    con.close()
    df["answerID"] = df["answerID"].astype(int)
    return df.set_index("answerID")

def load_benchmark(xlsx_path, sheet, id_min, id_max):
    df = pd.read_excel(xlsx_path, sheet_name=sheet, dtype=str)
    df["notegroupID"] = df["notegroupID"].astype(int)
    df["answerID"]    = df["answerID"].astype(int)
    df = df[df["notegroupID"].between(id_min, id_max)]
    return df.set_index("answerID")

def load_id_map(csv_path, etl_col, bm_col):
    """Load a matched-pairs CSV and return etl->bm and bm->etl id dicts."""
    df = pd.read_csv(csv_path)
    df[etl_col] = df[etl_col].astype(int)
    df[bm_col]  = df[bm_col].astype(int)
    etl_to_bm = dict(zip(df[etl_col], df[bm_col]))
    bm_to_etl = dict(zip(df[bm_col], df[etl_col]))
    return etl_to_bm, bm_to_etl

def load_matched_answers(csv_path):
    df = pd.read_csv(csv_path)
    df["etl_answerID"] = df["etl_answerID"].astype(int)
    df["bm_answerID"]  = df["bm_answerID"].astype(int)
    return df

etl     = load_etl(DB_PATH, NOTEGROUP_ID_MIN, NOTEGROUP_ID_MAX)
bm      = load_benchmark(BENCHMARK_PATH, BENCHMARK_SHEET, NOTEGROUP_ID_MIN, NOTEGROUP_ID_MAX)
matched = load_matched_answers(MATCHED_ANSWERS_CSV)

def load_etl_questions(db_path, id_min, id_max):
    con = sqlite3.connect(db_path)
    df = pd.read_sql_query(
        "SELECT questionID, question_content FROM questions WHERE notegroupID BETWEEN ? AND ?",
        con, params=(id_min, id_max)
    )
    con.close()
    df["questionID"] = df["questionID"].astype(int)
    return df.set_index("questionID")

def load_bm_questions(xlsx_path, id_min, id_max):
    df = pd.read_excel(xlsx_path, sheet_name="questions", dtype=str)
    df = df.dropna(subset=["notegroupID"])
    df["notegroupID"] = df["notegroupID"].astype(int)
    df["questionID"]  = df["questionID"].astype(int)
    df = df[df["notegroupID"].between(id_min, id_max)]
    return df.set_index("questionID")

etl_questions = load_etl_questions(DB_PATH, NOTEGROUP_ID_MIN, NOTEGROUP_ID_MAX)
bm_questions  = load_bm_questions(BENCHMARK_PATH, NOTEGROUP_ID_MIN, NOTEGROUP_ID_MAX)

participant_etl_to_bm, _ = load_id_map(MATCHED_PARTICIPANTS_CSV, "etl_participantID", "bm_participantID")
question_etl_to_bm, _    = load_id_map(MATCHED_QUESTIONS_CSV,    "etl_questionID",    "bm_questionID")

matched_etl = set(matched["etl_answerID"])
matched_bm  = set(matched["bm_answerID"])
etl_only    = sorted(set(etl.index) - matched_etl)
bm_only     = sorted(set(bm.index)  - matched_bm)

print(f"ETL records   : {len(etl)}")
print(f"BM records    : {len(bm)}")
print(f"Matched pairs : {len(matched)}")
print(f"ETL-only (FP) : {len(etl_only)}")
print(f"BM-only  (FN) : {len(bm_only)}")

ETL records   : 1218
BM records    : 1232
Matched pairs : 1204
ETL-only (FP) : 14
BM-only  (FN) : 28


In [3]:
def normalise_str(val):
    if pd.isna(val) or str(val).strip() in ("", "None", "nan"):
        return None
    s = str(val).strip().lower()
    s = re.sub(r'\s*\n\s*', '\n', s)
    return s

def normalise_id(val):
    if pd.isna(val) or str(val).strip() in ("", "None", "nan"):
        return None
    try:
        return int(float(str(val).strip()))
    except (ValueError, TypeError):
        return None

def normalise_mapped_id(val, id_map):
    """Translate an ETL id to BM space using id_map, return as string or None."""
    qid = normalise_id(val)
    if qid is None:
        return None
    mapped = id_map.get(qid)
    return str(mapped) if mapped is not None else str(qid)

def compute_counts(etl_val, bm_val, sim_threshold=None):
    """
    Return (TP, FP, FN, TN) for one field comparison.
    If sim_threshold is set, treat as a match when fuzz.ratio/100 >= sim_threshold
    instead of requiring exact equality.
    """
    e = normalise_str(etl_val)
    b = normalise_str(bm_val)
    if e is not None and b is not None:
        if sim_threshold is not None:
            is_match = (fuzz.ratio(e, b) / 100) >= sim_threshold
        else:
            is_match = (e == b)
        return (1, 0, 0, 0) if is_match else (0, 1, 1, 0)
    if e is not None and b is None:
        return (0, 1, 0, 0)
    if e is None and b is not None:
        return (0, 0, 1, 0)
    return (0, 0, 0, 1)

def safe_div(num, den):
    return round(num / den, 4) if den > 0 else None

def metrics_from_counts(TP, FP, FN, TN):
    accuracy  = safe_div(TP + TN, TP + FP + FN + TN)
    precision = safe_div(TP, TP + FP)
    recall    = safe_div(TP, TP + FN)
    f1 = round(2 * precision * recall / (precision + recall), 4) \
         if precision and recall and (precision + recall) > 0 else None
    return dict(TP=TP, FP=FP, FN=FN, TN=TN,
                accuracy=accuracy, precision=precision, recall=recall, F1=f1)

In [4]:
totals = {f: dict(TP=0, FP=0, FN=0, TN=0) for f in EVAL_FIELDS}

for _, row in matched.iterrows():
    ei = row["etl_answerID"]
    bi = row["bm_answerID"]
    for field in EVAL_FIELDS:
        etl_val = etl.at[ei, field] if field in etl.columns else None
        bm_val  = bm.at[bi, field]  if field in bm.columns  else None

        if field == "participantID":
            etl_val = normalise_mapped_id(etl_val, participant_etl_to_bm)
            bm_val  = normalise_str(str(bm_val)) if not pd.isna(bm_val) and str(bm_val).strip() not in ("", "None", "nan") else None
            tp, fp, fn, tn = compute_counts(etl_val, bm_val)
        elif field == "questionID":
            etl_qid = normalise_id(etl_val)
            bm_qid  = normalise_id(bm_val)
            # Try ID mapping first
            etl_mapped = normalise_mapped_id(etl_val, question_etl_to_bm)
            bm_id_str  = normalise_str(str(bm_val)) if not pd.isna(bm_val) and str(bm_val).strip() not in ("", "None", "nan") else None
            if etl_mapped is not None and bm_id_str is not None and etl_mapped == bm_id_str:
                tp, fp, fn, tn = (1, 0, 0, 0)
            else:
                # Fallback: compare question_content via partial_ratio
                etl_qc = normalise_str(etl_questions.at[etl_qid, "question_content"]) \
                         if etl_qid is not None and etl_qid in etl_questions.index else None
                bm_qc  = normalise_str(bm_questions.at[bm_qid,  "question_content"]) \
                         if bm_qid  is not None and bm_qid  in bm_questions.index  else None
                if etl_qc is not None and bm_qc is not None:
                    match = fuzz.partial_ratio(etl_qc, bm_qc) / 100 >= 0.85
                    tp, fp, fn, tn = (1, 0, 0, 0) if match else (0, 1, 1, 0)
                elif etl_qc is not None and bm_qc is None:
                    tp, fp, fn, tn = (0, 1, 0, 0)
                elif etl_qc is None and bm_qc is not None:
                    tp, fp, fn, tn = (0, 0, 1, 0)
                else:
                    tp, fp, fn, tn = (0, 0, 0, 1)
        elif field in ("answer_content_oriLAN", "answer_content_EN"):
            tp, fp, fn, tn = compute_counts(etl_val, bm_val, sim_threshold=CONTENT_SIM_THRESHOLD)
        else:
            tp, fp, fn, tn = compute_counts(etl_val, bm_val)

        totals[field]["TP"] += tp; totals[field]["FP"] += fp
        totals[field]["FN"] += fn; totals[field]["TN"] += tn

# ETL-only rows → every field counts as FP
for ei in etl_only:
    for field in EVAL_FIELDS:
        totals[field]["FP"] += 1

# BM-only rows → every field counts as FN
for bi in bm_only:
    for field in EVAL_FIELDS:
        totals[field]["FN"] += 1

In [5]:
# Write field-level agreement results to matched_answers.csv
def field_result(ei, bi, field):
    """Return 1=match, 0=mismatch, None=both null for a single field comparison."""
    etl_val = etl.at[ei, field] if field in etl.columns else None
    bm_val  = bm.at[bi, field]  if field in bm.columns  else None

    if field == "participantID":
        etl_val = normalise_mapped_id(etl_val, participant_etl_to_bm)
        bm_val  = normalise_str(str(bm_val)) if not pd.isna(bm_val) and str(bm_val).strip() not in ("", "None", "nan") else None
        tp, fp, fn, tn = compute_counts(etl_val, bm_val)

    elif field == "questionID":
        etl_qid = normalise_id(etl_val)
        bm_qid  = normalise_id(bm_val)
        etl_mapped = normalise_mapped_id(etl_val, question_etl_to_bm)
        bm_id_str  = normalise_str(str(bm_val)) if not pd.isna(bm_val) and str(bm_val).strip() not in ("", "None", "nan") else None
        if etl_mapped is not None and bm_id_str is not None and etl_mapped == bm_id_str:
            tp, fp, fn, tn = (1, 0, 0, 0)
        else:
            etl_qc = normalise_str(etl_questions.at[etl_qid, "question_content"]) \
                     if etl_qid is not None and etl_qid in etl_questions.index else None
            bm_qc  = normalise_str(bm_questions.at[bm_qid,  "question_content"]) \
                     if bm_qid  is not None and bm_qid  in bm_questions.index  else None
            if etl_qc is not None and bm_qc is not None:
                match = fuzz.partial_ratio(etl_qc, bm_qc) / 100 >= 0.85
                tp, fp, fn, tn = (1, 0, 0, 0) if match else (0, 1, 1, 0)
            elif etl_qc is not None and bm_qc is None:
                tp, fp, fn, tn = (0, 1, 0, 0)
            elif etl_qc is None and bm_qc is not None:
                tp, fp, fn, tn = (0, 0, 1, 0)
            else:
                tp, fp, fn, tn = (0, 0, 0, 1)

    elif field in ("answer_content_oriLAN", "answer_content_EN"):
        tp, fp, fn, tn = compute_counts(etl_val, bm_val, sim_threshold=CONTENT_SIM_THRESHOLD)

    else:
        tp, fp, fn, tn = compute_counts(etl_val, bm_val)

    if tn == 1:
        return None
    return 1 if tp == 1 else 0

# Load existing CSV and append result columns
matched_csv = pd.read_csv(MATCHED_ANSWERS_CSV)
matched_csv["etl_answerID"] = matched_csv["etl_answerID"].astype(int)
matched_csv["bm_answerID"]  = matched_csv["bm_answerID"].astype(int)

for field in EVAL_FIELDS:
    matched_csv[f"result_{field}"] = matched_csv.apply(
        lambda row: field_result(row["etl_answerID"], row["bm_answerID"], field),
        axis=1
    )

matched_csv.to_csv(MATCHED_ANSWERS_CSV, index=False)
print(f"Updated {MATCHED_ANSWERS_CSV} with result columns for {EVAL_FIELDS}")

Updated matched_answers.csv with result columns for ['participantID', 'questionID', 'answer_content_oriLAN', 'answer_content_EN']


In [6]:
rows = []
for field in EVAL_FIELDS:
    m = metrics_from_counts(**totals[field])
    rows.append({"field": field, **m})

overall = {k: sum(totals[f][k] for f in EVAL_FIELDS) for k in ("TP","FP","FN","TN")}
m_all = metrics_from_counts(**overall)
rows.append({"field": "OVERALL", **m_all})

results_df = pd.DataFrame(rows).set_index("field")
results_df

,TP,FP,FN,TN,accuracy,precision,recall,F1
field,,,,,,,,
participantID,1122,38,48,58,0.9321,0.9672,0.9590,0.9631
questionID,1203,15,29,0,0.9647,0.9877,0.9765,0.9821
answer_content_oriLAN,1190,28,42,0,0.9444,0.9770,0.9659,0.9714
answer_content_EN,109,15,29,1094,0.9647,0.8790,0.7899,0.8321
OVERALL,3624,96,148,1152,0.9514,0.9742,0.9608,0.9675
